# Python 7: errors, logging and testing

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python/learning/python-07-errors-logging-and-testing.ipynb)

A pipeline that fails silently is worse than one that fails loudly. This chapter covers the three habits that make Python code trustworthy in production. First, raise specific errors, and handle only the errors you can really handle. Second, log what happened in a form people can search. Third, test transformations with small, fast checks that run on every change.

Every rule is stated before the code that shows it, and each code cell's `assert` lines are the expected results. If an `assert` fails, your model of that rule is wrong.
**Runtime:** Python 3.12 or later.

## Exceptions: the hierarchy

An **exception** is an object that describes an error. Raising it stops the current block and unwinds the call stack until some `except` handles it. Exceptions form a class hierarchy. `except SomeError` catches that class and every subclass of it.

Two base classes matter. `Exception` is the parent of the errors your code should handle. `BaseException` is the parent of the exceptions that should usually stop the program, such as `KeyboardInterrupt`, which is Ctrl-C. A plain `except Exception` does not catch Ctrl-C, but a bare `except:` does, which is why a bare `except:` is almost always wrong.

In [ ]:
print(f"{issubclass(ValueError, Exception) = }")
print(f"{issubclass(KeyboardInterrupt, BaseException) = }")
print(f"{not issubclass(KeyboardInterrupt, Exception) = }")  # except Exception lets Ctrl-C through

**What to notice:** `ValueError` is an `Exception`, but `KeyboardInterrupt` is not: it derives directly from `BaseException`. That is why `except Exception` does not swallow Ctrl+C, and a bare `except:` does.

## `try`, `except`, `else` and `finally`

- `try` holds the code that may fail.
- `except` handles one class of exception. Catch the narrowest class that you expect.
- `else` runs only when the `try` block raised nothing.
- `finally` runs **every time**, including after a `return` or a raised error. Use it for cleanup that must happen.

A `return` inside `try` skips `else`, so put the return in `else` when you need `else` to run first.

In [ ]:
trace = []

def read_count(text):
    try:
        value = int(text)
    except ValueError:
        trace.append("except")
        return None
    else:
        trace.append("else")
        return value
    finally:
        trace.append("finally")

print(f"{read_count('5') = }")
print(f"{trace = }")  # the else ran, then finally ran after the return
trace.clear()
print(f"{read_count('x') = }")
print(f"{trace = }")  # finally runs even when the except returns

**What to notice:** With valid input the `else` block ran, then `finally`; with bad input the `except` block ran, then `finally`. `finally` runs either way, which makes it the place for clean-up.

### Where you'll use it

- **Transient network errors:** catch `TimeoutError` and retry. Let every other error stop the job, so a real bug is never hidden.
- **Parsing with a fallback:** catch the one error that means "bad text", and count the row as quarantined.

In [ ]:
def parse_amount(text):
    try:
        return float(text)
    except ValueError:                       # expected: the source sent bad text
        return None

print(f"{parse_amount('12.5') = }")
print(f"{parse_amount('n/a') = }")

**What to notice:** Catching only `ValueError` turns unparseable text into `None`, and lets any other error propagate.

## Raising and chaining errors

Raise a specific error with a clear message. Define your own exception classes for the failures your pipeline cares about, and give them a common base class. Callers can then catch `PipelineError` for everything, or a subclass for one failure.

When you convert a low-level error into a higher-level one, use `raise ... from error`. The original error is kept as `__cause__`, and the traceback shows both. Use `raise ... from None` only when the original error is noise.

### Where you'll use it

- **Schema contracts:** a `SchemaError` raised by the loader tells the on-call engineer exactly which columns drifted.
- **Adding context:** a driver error is wrapped with the file name and row number, while the original traceback is kept.

In [ ]:
class PipelineError(Exception):
    """Base class for every error this pipeline raises on purpose."""

class SchemaError(PipelineError):
    """A record does not match the expected columns."""

def require_columns(record, required):
    missing = [column for column in required if column not in record]
    if missing:
        raise SchemaError(f"missing columns: {', '.join(missing)}")
    return record

try:
    require_columns({"id": 1}, ["id", "amount"])
except PipelineError as error:              # the base class catches the subclass
    message = str(error)
    caught = type(error).__name__

print(f"{message = }")
print(f"{caught = }")

def load_amount(record):
    try:
        return float(record["amount"])
    except (KeyError, ValueError) as error:
        raise SchemaError("bad amount") from error

try:
    load_amount({"amount": "x"})
except SchemaError as error:
    print(f"{isinstance(error.__cause__, ValueError) = }")  # the original cause is kept

try:
    try:
        {}["key"]
    except KeyError:
        raise LookupError("no such field") from None
except LookupError as error:
    print(f"{error.__cause__ = }")  # the KeyError is hidden on purpose

**What to notice:** The custom `SchemaError` carries a clear message and can be caught as a `PipelineError`. `raise ... from error` stores the original `ValueError` in `__cause__`; `from None` hides it on purpose.

## Logging instead of `print`

The `logging` module records messages with a **level**, a time and a **logger name**. Levels let you filter: `DEBUG` for detail, `INFO` for progress, `WARNING` for something odd, and `ERROR` for a failed step. A logger passes on only the messages at or above its level.

Pass values as arguments with `%s` or `%d` placeholders, not as f-strings. The message is built only when the record is actually emitted, so a `DEBUG` line that is filtered out costs almost nothing. Inside an `except` block, `logger.exception` logs the message and the traceback.

### Where you'll use it

- **Operating pipelines:** logs are how an engineer learns why a 3 a.m. run loaded zero rows. INFO lines with counts answer that question.
- **Alerting:** a monitoring tool counts `ERROR` lines and pages the on-call engineer when they pass a threshold.

In [ ]:
import io
import json
import logging

stream = io.StringIO()
logger = logging.getLogger("orders.load")
logger.handlers.clear()
handler = logging.StreamHandler(stream)
handler.setFormatter(logging.Formatter("%(levelname)s %(name)s: %(message)s"))
logger.addHandler(handler)
logger.setLevel(logging.INFO)

logger.debug("hidden: below the level")
logger.info("loaded %d rows from %s", 120, "orders.csv")
logger.warning("skipped %d bad rows", 3)

try:
    load_amount({"amount": "x"})
except SchemaError:
    logger.exception("could not load a record")

# A structured line: one JSON object after the message, easy to parse in a log search tool.
logger.info("run finished %s", json.dumps({"run_id": "r1", "loaded": 120}, sort_keys=True))

lines = stream.getvalue().splitlines()
print(f"{lines[0] = }")
print(f"{lines[1] = }")
print(f"{lines[2].startswith('ERROR orders.load: could not load a record') = }")
print(f"{not any(('hidden' in line for line in lines)) = }")
print(f"{lines[-1] = }")

**What to notice:** Each log line carries the level and the logger name, so `WARNING` and `ERROR` lines are easy to find. The `DEBUG` message was hidden by the level, `logger.exception` added the traceback, and `extra` fields attach structured context to the last line.

In [ ]:
class Expensive:
    """Counts how often its text is built."""
    calls = 0

    def __str__(self):
        Expensive.calls += 1
        return "expensive"

logger.debug("state: %s", Expensive())    # filtered out at INFO, so str() never runs
print(f"{Expensive.calls = }")

logger.info("state: %s", Expensive())     # emitted, so str() runs once
print(f"{Expensive.calls = }")

**What to notice:** With the level set to `WARNING`, the `DEBUG` call never formatted its argument: `Expensive.calls` stayed 0. Pass arguments with `%s` instead of building an f-string, so the work is skipped when the message is not logged.

## Testing transformations

A transformation is a function from input to output, which makes it easy to test. Test the behaviour you care about: the normal case, the empty input, the edge values and the errors. Keep each test small and independent, so a failure points at one rule.

In a notebook, a test is a function whose name starts with `test_` and whose body is `assert` statements. A test runner such as `pytest` finds every `test_` function in a file and reports each failure with its own message. The plain loop below does the same job for this notebook.

### Where you'll use it

- **CI for data code:** a test suite runs on every pull request, so a change to a parsing rule breaks the build instead of production.
- **Regression tests:** when a bad record reaches production, add it as a test case, so the bug cannot come back unnoticed.

In [ ]:
def normalise_country(code):
    if not code or not code.strip():
        raise ValueError("country code is empty")
    return code.strip().upper()

# Table-driven: one list of (input, expected) pairs, one test function.
COUNTRY_CASES = [
    (" de ", "DE"),
    ("nl", "NL"),
    ("Us", "US"),
    ("\tfr\n", "FR"),              # regression case: tabs and newlines from a source file
]

def test_normalise_country_strips_and_uppercases():
    for raw, expected in COUNTRY_CASES:
        assert normalise_country(raw) == expected, f"input {raw!r}"

def test_normalise_country_rejects_blank():
    try:
        normalise_country("   ")
    except ValueError:
        return
    raise AssertionError("expected ValueError")

def test_normalise_country_is_idempotent():
    once = normalise_country("nl")
    assert normalise_country(once) == once

tests = [obj for name, obj in list(globals().items()) if name.startswith("test_") and callable(obj)]
for test in tests:
    test()
    print("PASS", test.__name__)

# An assertion can carry a message. The message appears in the failure report.
def assert_codes_are_lowercase():  # a deliberately wrong expectation, to see a failure
    assert normalise_country("de") == "de", "codes are stored in uppercase"

try:
    assert_codes_are_lowercase()
except AssertionError as error:
    print("FAIL assert_codes_are_lowercase:", error)

**What to notice:** All three tests passed. The deliberately wrong expectation failed with its message, `codes are stored in uppercase`, which is what a test runner prints next to the failing test.

## Measuring time

`time.perf_counter()` is the clock to use for durations. It is monotonic, so it never goes backwards, and its resolution is high. `time.time()` reads the wall clock, which can jump when the system clock is adjusted. A context manager makes timing reusable.

### Where you'll use it

- **Finding slow steps:** timing each stage of a job shows which one to optimise first.
- **Benchmarks:** comparing two parsing approaches on a sample file before you choose one.

In [ ]:
import time
from contextlib import contextmanager

@contextmanager
def timed(label, log):
    start = time.perf_counter()
    try:
        yield
    finally:
        log.append((label, time.perf_counter() - start))

timings = []
with timed("sum", timings):
    sum(range(1_000_000))
print(f"{timings[0][0] = }")
print(f"{timings[0][1] >= 0 = }")

**What to notice:** `time.perf_counter` measures elapsed time with the best resolution available; recording `(name, seconds)` pairs is the simplest profiling there is.

## Modules, imports and the `__main__` guard

A **module** is one `.py` file; a **package** is a directory of modules, marked by `__init__.py`. `import math` binds the module object, and `from decimal import Decimal` binds one name from it. Python finds modules on `sys.path`, runs each one once per process and caches it in `sys.modules`. Module-level code therefore runs on the first import, which is why heavy work at import time (opening connections, reading files) is a bug: Airflow, pytest and every other importer pay for it.

A typical pipeline package:

```text
src/etl_jobs/
  __init__.py
  parsers.py      # pure functions, easy to unit-test
  run.py          # thin entry point with the __main__ guard
```

The example builds a package in a temporary folder and imports it. Changing `sys.path` is only needed because the demo lives in a temporary folder; real code installs the package (`uv sync`) instead.

### Where you'll use it

- **Testable code:** keeping logic in importable modules lets pytest import and test it without running the job.
- **Airflow DAG files:** the scheduler imports them every few seconds, so anything slow at module level slows the whole scheduler.

In [ ]:
import importlib
import sys
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    package = Path(tmp) / "etl_jobs"
    package.mkdir()
    (package / "__init__.py").write_text("")
    (package / "parsers.py").write_text("def parse(raw):\n    return raw['id'].strip().upper()\n")

    sys.path.insert(0, tmp)
    try:
        parsers = importlib.import_module("etl_jobs.parsers")
        print("parsers.parse({'id': ' a1 '})", '->', parsers.parse({'id': ' a1 '}))
        print(f"{importlib.import_module('etl_jobs.parsers') is parsers = }")  # cached
        print(f"{sys.modules['etl_jobs.parsers'] is parsers = }")
    finally:
        sys.path.remove(tmp)
        for name in ["etl_jobs.parsers", "etl_jobs"]:
            sys.modules.pop(name, None)

**What to notice:** Importing a module runs it once and caches it in `sys.modules`: a second import returns the same module object.

### The `if __name__ == "__main__":` guard

Python sets `__name__` to `"__main__"` for the file you run directly, and to the module's dotted name when the file is imported. The guard keeps the entry point (argument parsing, the actual run) from firing when a test or a DAG file imports the module. In a notebook, the kernel itself is `__main__`.

In [ ]:
import importlib
import sys
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    (Path(tmp) / "run_job.py").write_text(
        "RAN = []\n"
        "def main():\n"
        "    RAN.append('main')\n"
        "if __name__ == '__main__':\n"
        "    main()\n"
    )
    sys.path.insert(0, tmp)
    try:
        run_job = importlib.import_module("run_job")
        print(f"{run_job.__name__ = }")
        print(f"{run_job.RAN = }")  # importing did not run main()
        run_job.main()
        print(f"{run_job.RAN = }")  # callers run it explicitly
    finally:
        sys.path.remove(tmp)
        sys.modules.pop("run_job", None)

print(f"{__name__ = }")  # the notebook kernel is __main__

**What to notice:** Importing `run_job` did not run its `main`, because `__name__` was the module name; calling the guarded block with `__name__ == "__main__"` ran it. The guard lets one file be both an importable module and a script.

## Interview notes

- **Catch narrowly and raise specifically.** Every `except Exception: pass` is a bug waiting to hide data loss.
- **Keep the cause.** `raise ... from error` shows both errors in the traceback, which speeds up debugging.
- **`finally` runs after `return`.** Use it for cleanup, and `else` for code that should run only when nothing failed.
- **Log with `%s` placeholders and levels.** Logs are data: searchable, filterable and the first thing read after an incident.
- **Test behaviour, not implementation.** A good test still passes after a refactor that keeps the behaviour the same.
- **Make every bug a test.** Add the failing input to the test table when you fix it, so the regression stays fixed.
- **No side effects at import time.** Modules run once on first import; keep the entry point behind `if __name__ == "__main__":`.

## Check yourself

Write your answer in the empty cell under each question, run it, then open the answer to compare.

**Q1.** Write `safe_int(text, default)` that returns the integer, or `default` when the text is not an integer. Catch only the errors that `int` raises for bad input.

In [ ]:
# Write your code here.

In [ ]:
def safe_int(text, default):
    try:
        return int(text)
    except (TypeError, ValueError):
        return default

print(f"{safe_int('42', 0) = }")
print(f"{safe_int('x', -1) = }")
print(f"{safe_int(None, 0) = }")

**Q2.** Write `call_with_retries(fn, attempts, state)`. The function `flaky(state)` raises `ConnectionError` on its first two calls and returns `"ok"` after that. Call it up to `attempts` times. Return the first result, or re-raise the last error.

In [ ]:
# Write your code here.

In [ ]:
def call_with_retries(fn, attempts, state):
    last_error = None
    for _ in range(attempts):
        try:
            return fn(state)
        except ConnectionError as error:
            last_error = error
    raise last_error

def flaky(state):
    state["calls"] += 1
    if state["calls"] <= 2:
        raise ConnectionError("temporary")
    return "ok"

state = {"calls": 0}
print(f"{call_with_retries(flaky, 3, state) = }")
print(f"{state['calls'] = }")

**Q3.** Write a test function `test_safe_int_returns_default_for_decimal_text` that checks `safe_int("12.5", 0)` returns `0`, and run it.

In [ ]:
# Write your code here.

In [ ]:
def test_safe_int_returns_default_for_decimal_text():
    print(f"{safe_int('12.5', 0) = }")

test_safe_int_returns_default_for_decimal_text()

## Key takeaways

- Catch the specific exceptions you can handle; let the rest propagate.
- `else` runs when no exception happened, `finally` always runs; chain errors with `raise ... from`.
- Log with the `logging` module and levels instead of `print`.
- Test transformations with small table-driven test functions: normal, empty, edge and error cases.
- Measure time with `time.perf_counter`, and guard script entry points with `if __name__ == "__main__"`.

**Next:** Practise this lesson in [Python drills 7: errors, logging and testing](../practice/python-drills-07-errors-logging-and-testing.ipynb), then continue with [Python 8: concurrency](python-08-concurrency.ipynb).